# Task 1.3 - Math for ML Practice Set (worked solutions)
**Skill Set Go EduTech - AI/ML Internship, Week 1**

Every problem below is solved **by hand with the steps written out**, then verified
numerically with NumPy, and closed with one line on *where the same idea appears
inside an ML algorithm*. Final answers on their own are not the point.

Sections: 1 descriptive statistics - 2 probability - 3 linear algebra -
4 calculus and gradient descent.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
print("numpy", np.__version__)

## Section 1 - Descriptive statistics

### Q1.1 Mean, median, variance, standard deviation
Marks of 8 students: `x = [62, 71, 68, 90, 74, 71, 55, 89]`

**Step 1 - mean.**
$\bar{x} = \frac{62+71+68+90+74+71+55+89}{8} = \frac{580}{8} = 72.5$

**Step 2 - median.** Sorted: 55, 62, 68, 71, 71, 74, 89, 90. With n = 8 (even) the
median is the average of the 4th and 5th values: $(71+71)/2 = 71$.

**Step 3 - deviations and squared deviations.**

| x | x - x̄ | (x - x̄)² |
|---|---|---|
| 62 | -10.5 | 110.25 |
| 71 | -1.5 | 2.25 |
| 68 | -4.5 | 20.25 |
| 90 | 17.5 | 306.25 |
| 74 | 1.5 | 2.25 |
| 71 | -1.5 | 2.25 |
| 55 | -17.5 | 306.25 |
| 89 | 16.5 | 272.25 |
| **sum** | **0** | **1022** |

**Step 4 - variance.** Population: $\sigma^2 = 1022/8 = 127.75$.
Sample: $s^2 = 1022/7 = 146.0$ (divide by n-1 because the mean was estimated from
the same data, which uses up one degree of freedom).

**Step 5 - standard deviation.** $\sigma = \sqrt{127.75} \approx 11.30$ marks - back
in the original units, which is why it is reported instead of variance.

**ML link.** This is exactly what `StandardScaler` computes: it subtracts the mean
and divides by the standard deviation so that features measured in different units
contribute comparably to a distance or a gradient.

In [ ]:
x = np.array([62, 71, 68, 90, 74, 71, 55, 89], dtype=float)
print("mean            :", x.mean())
print("median          :", np.median(x))
print("variance (pop)  :", x.var())
print("variance (sample):", x.var(ddof=1))
print("std dev (pop)   :", round(x.std(), 4))
print("deviations sum to zero:", round((x - x.mean()).sum(), 10))
print("standardised    :", (x - x.mean()) / x.std())

### Q1.2 Covariance and correlation
Hours studied `h = [2, 4, 6, 8, 10]`, marks `m = [55, 60, 70, 78, 87]`

**Step 1 - means.** $\bar{h} = 6$, $\bar{m} = 70$.

**Step 2 - products of deviations.**

| h | m | h-h̄ | m-m̄ | product |
|---|---|---|---|---|
| 2 | 55 | -4 | -15 | 60 |
| 4 | 60 | -2 | -10 | 20 |
| 6 | 70 | 0 | 0 | 0 |
| 8 | 78 | 2 | 8 | 16 |
| 10 | 87 | 4 | 17 | 68 |
| | | | **sum** | **164** |

**Step 3 - covariance (sample).** $\text{cov} = 164/(5-1) = 41$.
Positive, so the two move in the same direction - but the size 41 is meaningless on
its own because it depends on the units.

**Step 4 - correlation.** $s_h = \sqrt{40/4} = \sqrt{10} \approx 3.162$,
$s_m = \sqrt{678/4} = \sqrt{169.5} \approx 13.019$.
$r = \frac{41}{3.162 \times 13.019} \approx 0.996$

**Step 5 - interpretation.** r ≈ 0.996 is a near-perfect *linear* association.
It does **not** prove that studying causes higher marks: a third factor such as prior
preparation could drive both, and with only 5 points the estimate is fragile.

**ML link.** The correlation matrix in Task 1.2 was this calculation applied pairwise;
two features with r near 1 carry duplicate information for a linear model.

In [ ]:
h = np.array([2, 4, 6, 8, 10], dtype=float)
m = np.array([55, 60, 70, 78, 87], dtype=float)
cov = ((h - h.mean()) * (m - m.mean())).sum() / (len(h) - 1)
print("covariance (sample):", cov)
print("numpy cov matrix   :\n", np.cov(h, m))
print("correlation        :", round(np.corrcoef(h, m)[0, 1], 4))

## Section 2 - Probability

### Q2.1 Conditional probability and independence
A model is tested on 200 images: 120 cats, 80 dogs. It predicts "cat" for 130 images,
and 110 of those are genuinely cats.

**Step 1 - the counts.**

| | predicted cat | predicted dog | total |
|---|---|---|---|
| actually cat | 110 (TP) | 10 (FN) | 120 |
| actually dog | 20 (FP) | 60 (TN) | 80 |
| **total** | **130** | **70** | **200** |

**Step 2 - P(actually cat | predicted cat)** = 110/130 ≈ 0.846. This is **precision**.

**Step 3 - P(predicted cat | actually cat)** = 110/120 ≈ 0.917. This is **recall**.
The two conditionals are different numbers because the condition differs - confusing
them is one of the most common mistakes in model reporting.

**Step 4 - independence check.** P(cat) = 120/200 = 0.6 and
P(predicted cat) = 130/200 = 0.65, so if the events were independent
P(cat ∩ predicted cat) would be 0.6 × 0.65 = 0.39, i.e. 78 images.
The observed count is 110, far higher - so the prediction is (fortunately) *not*
independent of the truth.

**ML link.** This is the confusion matrix from Task 1.4 written as probabilities;
precision, recall and F1 are all conditional probabilities read off this table.

In [ ]:
TP, FN, FP, TN = 110, 10, 20, 60
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
print(f"precision = {precision:.4f}")
print(f"recall    = {recall:.4f}")
print(f"F1        = {f1:.4f}")
print(f"accuracy  = {(TP + TN) / 200:.4f}")
print("independent-case count would be:", 0.6 * 0.65 * 200, "vs observed", TP)

### Q2.2 Bayes' theorem
A disease affects 1% of a population. A test detects it in 99% of sick people
(sensitivity) and wrongly flags 5% of healthy people (false-positive rate).
**A person tests positive - what is the probability they are actually sick?**

**Step 1 - write the knowns.** P(D) = 0.01, P(+|D) = 0.99, P(+|¬D) = 0.05.

**Step 2 - total probability of a positive test.**
$P(+) = 0.99 \times 0.01 + 0.05 \times 0.99 = 0.0099 + 0.0495 = 0.0594$

**Step 3 - Bayes.**
$P(D|+) = \frac{P(+|D)P(D)}{P(+)} = \frac{0.0099}{0.0594} \approx 0.1667$

**Step 4 - interpretation.** Only about **17%**. Because the disease is rare, the
5% of false positives drawn from the large healthy group outnumber the true positives
5 to 1. A 99% sensitive test is still mostly wrong when it fires on a rare condition.

**ML link.** The same arithmetic explains why a classifier with high accuracy on an
imbalanced dataset can still be useless, and it is the rule Naive Bayes applies to
turn P(features|class) into P(class|features).

In [ ]:
p_d, p_pos_given_d, p_pos_given_not_d = 0.01, 0.99, 0.05
p_pos = p_pos_given_d * p_d + p_pos_given_not_d * (1 - p_d)
print(f"P(+)     = {p_pos:.4f}")
print(f"P(D | +) = {p_pos_given_d * p_d / p_pos:.4f}")
for prevalence in (0.001, 0.01, 0.1, 0.5):
    pp = 0.99 * prevalence + 0.05 * (1 - prevalence)
    print(f"  prevalence {prevalence:<6} -> P(D|+) = {0.99 * prevalence / pp:.4f}")

### Q2.3 Expected value
A model makes a prediction that is correct with probability 0.8. A correct call gains
10 units, a wrong call costs 25 units. **Should it act on the prediction?**

$E[\text{payoff}] = 0.8 \times 10 + 0.2 \times (-25) = 8 - 5 = 3$

Positive, so acting is worth it on average. The break-even accuracy p solves
$10p - 25(1-p) = 0 \Rightarrow 35p = 25 \Rightarrow p \approx 0.714$ - below about 71%
accuracy the system loses money even though it is "usually right".

**ML link.** This is cost-sensitive evaluation: the right decision threshold depends
on the cost of each error type, not on accuracy alone.

In [ ]:
gain, loss = 10, -25
for p in (0.7, 0.714, 0.8, 0.9):
    print(f"p={p:<6} E[payoff] = {p * gain + (1 - p) * loss:+.3f}")

## Section 3 - Linear algebra

### Q3.1 Dot product and why it is a weighted sum
$a = [2, -1, 3]$, $b = [4, 5, -2]$

**Step 1.** $a \cdot b = (2)(4) + (-1)(5) + (3)(-2) = 8 - 5 - 6 = -3$

**Step 2 - magnitudes.** $\|a\| = \sqrt{4+1+9} = \sqrt{14} \approx 3.742$,
$\|b\| = \sqrt{16+25+4} = \sqrt{45} \approx 6.708$

**Step 3 - angle.** $\cos\theta = \frac{-3}{3.742 \times 6.708} \approx -0.1195$,
so $\theta \approx 96.9°$ - slightly more than a right angle, meaning the vectors are
close to unrelated in direction.

**ML link.** A single neuron computes exactly $w \cdot x + b$. The dot product *is*
the weighted sum, and cosine similarity is this same quantity normalised - the metric
a vector store uses in Week 4's RAG task to rank retrieved chunks.

In [ ]:
a = np.array([2, -1, 3]); b = np.array([4, 5, -2])
dot = a @ b
cos = dot / (np.linalg.norm(a) * np.linalg.norm(b))
print("a . b        :", dot)
print("|a|, |b|     :", round(np.linalg.norm(a), 4), round(np.linalg.norm(b), 4))
print("cos(theta)   :", round(cos, 4))
print("theta (deg)  :", round(np.degrees(np.arccos(cos)), 2))

### Q3.2 Matrix multiplication and dimensions
$A = \begin{bmatrix} 1 & 2 \\ 3 & 4 \\ 5 & 6 \end{bmatrix}$ (3×2),
$B = \begin{bmatrix} 7 & 8 & 9 \\ 10 & 11 & 12 \end{bmatrix}$ (2×3)

**Step 1 - check dimensions.** (3×2)(2×3): the inner dimensions match (2 = 2), so the
product exists and is **3×3**. The reverse product BA is 2×2 - matrix multiplication
is not commutative.

**Step 2 - compute row by row.** Entry (i,j) is row i of A dotted with column j of B.
- (1,1): 1(7) + 2(10) = 27  - (1,2): 1(8) + 2(11) = 30  - (1,3): 1(9) + 2(12) = 33
- (2,1): 3(7) + 4(10) = 61  - (2,2): 3(8) + 4(11) = 68  - (2,3): 3(9) + 4(12) = 75
- (3,1): 5(7) + 6(10) = 95  - (3,2): 5(8) + 6(11) = 106 - (3,3): 5(9) + 6(12) = 117

$AB = \begin{bmatrix} 27 & 30 & 33 \\ 61 & 68 & 75 \\ 95 & 106 & 117 \end{bmatrix}$

**ML link.** A batch of 3 samples with 2 features multiplied by a 2→3 weight matrix
produces 3 samples × 3 outputs. Every "shape mismatch" error in PyTorch next week is
this inner-dimension rule being violated.

In [ ]:
A = np.array([[1, 2], [3, 4], [5, 6]])
B = np.array([[7, 8, 9], [10, 11, 12]])
print("A", A.shape, "B", B.shape, "-> AB", (A @ B).shape)
print(A @ B)
print("\nBA is a different shape:", (B @ A).shape)
print(B @ A)

### Q3.3 A linear layer by hand
One sample $x = [1.0, 2.0, 3.0]$, weights $W = \begin{bmatrix} 0.2 & -0.1 & 0.4 \\ 0.5 & 0.3 & -0.2 \end{bmatrix}$ (2×3), bias $b = [0.1, -0.2]$.

**Step 1 - first output.** $0.2(1) + (-0.1)(2) + 0.4(3) = 0.2 - 0.2 + 1.2 = 1.2$; add bias 0.1 → **1.3**

**Step 2 - second output.** $0.5(1) + 0.3(2) + (-0.2)(3) = 0.5 + 0.6 - 0.6 = 0.5$; add bias −0.2 → **0.3**

**Step 3 - ReLU.** Both are positive, so ReLU leaves them unchanged: [1.3, 0.3].

**ML link.** This is one forward pass through one layer - the operation Week 2 repeats
thousands of times per epoch.

In [ ]:
xv = np.array([1.0, 2.0, 3.0])
W = np.array([[0.2, -0.1, 0.4], [0.5, 0.3, -0.2]])
bv = np.array([0.1, -0.2])
z = W @ xv + bv
print("z          :", z)
print("relu(z)    :", np.maximum(z, 0))

## Section 4 - Calculus and gradient descent

### Q4.1 Derivatives as a rate of change
$f(x) = 3x^2 - 4x + 7$

**Step 1 - differentiate term by term.** $f'(x) = 6x - 4$.

**Step 2 - evaluate.** $f'(2) = 8$: near x = 2 the function rises 8 units per unit of x.
$f'(0) = -4$: at x = 0 it is *falling*.

**Step 3 - minimum.** Set $f'(x) = 0 \Rightarrow 6x = 4 \Rightarrow x = 2/3$.
$f(2/3) = 3(4/9) - 8/3 + 7 = 4/3 - 8/3 + 7 = 16/3 \approx 5.333$.

**ML link.** The gradient is only a *direction and a rate*; training works by
repeatedly stepping against it, which is why a zero gradient means "stop here".

In [ ]:
f = lambda t: 3 * t**2 - 4 * t + 7
fp = lambda t: 6 * t - 4
for t in (0, 2 / 3, 2):
    numeric = (f(t + 1e-6) - f(t - 1e-6)) / 2e-6   # numeric check of the hand result
    print(f"x={t:<6.3f} f(x)={f(t):<8.4f} f'(x)={fp(t):<8.4f} numeric={numeric:.4f}")

### Q4.2 Gradient descent by hand on MSE
One feature, one weight, no bias: $\hat{y} = wx$, loss $L = (wx - y)^2$.
Data point x = 2, y = 6. Start at w = 0, learning rate η = 0.1.

**Step 1 - the gradient.** $\frac{dL}{dw} = 2(wx - y)\cdot x$.

**Step 2 - iteration 1.** w = 0 → prediction 0, error −6,
gradient $2(-6)(2) = -24$, update $w = 0 - 0.1(-24) = 2.4$.

**Step 3 - iteration 2.** prediction 4.8, error −1.2,
gradient $2(-1.2)(2) = -4.8$, update $w = 2.4 + 0.48 = 2.88$.

**Step 4 - iteration 3.** prediction 5.76, error −0.24, gradient −0.96, w = 2.976.

The weight is converging on w = 3, which gives the exact prediction 6. Each step
shrinks the error by a factor of 0.2 here, because η = 0.1 with x² = 4 gives
$1 - 2\eta x^2 = 0.2$.

**Step 5 - why the learning rate matters.** If η were 0.3 the factor becomes
$1 - 2(0.3)(4) = -1.4$ - the error flips sign and *grows* each step. That is exactly
the "loss went to NaN" failure mode, and it is why Week 2 asks for a learning-rate
experiment rather than a guess.

In [ ]:
def gradient_descent(lr, steps=8, w=0.0, xi=2.0, yi=6.0):
    history = []
    for i in range(steps):
        pred = w * xi
        loss = (pred - yi) ** 2
        grad = 2 * (pred - yi) * xi
        history.append((i, round(w, 6), round(loss, 6), round(grad, 6)))
        w = w - lr * grad
    return w, history

for lr in (0.1, 0.01, 0.3):
    final, hist = gradient_descent(lr)
    print(f"\nlearning rate {lr}")
    print(f"{'step':>4} {'w':>12} {'loss':>14} {'gradient':>14}")
    for row in hist[:5]:
        print(f"{row[0]:>4} {row[1]:>12} {row[2]:>14} {row[3]:>14}")
    print(f"  after 8 steps w = {final:.6f}  (target w = 3)")

**Reading the output.** η = 0.1 converges quickly to 3, η = 0.01 heads the right way
but far too slowly, and η = 0.3 diverges with the loss exploding - the three training
behaviours the Week 2 hyperparameter task asks me to produce deliberately.

## Summary - where each idea shows up in ML

| Concept | Where it appears |
|---|---|
| mean / std | feature scaling (`StandardScaler`), batch normalisation |
| variance & skew | choosing median over mean when imputing (done in Task 1.2) |
| covariance / correlation | redundant-feature detection, PCA |
| conditional probability | precision, recall, F1, confusion matrix |
| Bayes' theorem | Naive Bayes; why accuracy misleads on imbalanced data |
| expected value | cost-sensitive thresholds |
| dot product | a single neuron; cosine similarity in RAG retrieval |
| matrix multiplication | batched forward passes; every shape-mismatch error |
| derivative / gradient | backpropagation |
| gradient descent | every optimizer step, and what the learning rate controls |